# Table 3 and Gaussian Count Experiments

This notebook contains two independent experiments:

1. Table 3: full runtime, model size, and allocated CUDA memory peak through training.
2. Gaussian counts after density control on five synthetic samples and five real-event windows.

Every benchmark starts with CPU events. Reported memory includes upload, preprocessing, initialization and training, and excludes final rendering. Runtime still includes final rendering and detached CPU field outputs. GS uses fused CUDA rendering at native resolution (32-tile fallback batches). INR keeps full-image training and final velocity queries; final analytic derivative maps are optional and disabled by default.

The experiment cells display tables; this publication snapshot also contains CSV/JSON exports of their executed outputs.


Saved-result preview: Table 3 is restored from the complete verification saved on 2026-11-06 at 10:47 (UTC+08:00). The count tables are restored from the previously saved notebook outputs. No optimization was rerun to restore this display. These are historical saved results; newer shared-device measurements are retained separately in the result snapshot.


In [ ]:
import gc
import importlib
import math
import random
import sys
import time
from contextlib import ExitStack
from dataclasses import replace
from unittest.mock import patch
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    known_root = Path('/media/newdisk/junjie/2dGS-representation')
    for path in [known_root, start, *start.parents]:
        if (path / 'generator').is_dir() and (path / 'methods').is_dir():
            return path
    raise RuntimeError('Cannot find the project root.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generator.flow import FlowManager
from generator.images import config_events
from generator.loader import EventManager
# Reload dependencies first: an existing notebook kernel may still hold the
# renderer adapter and CUDA wrapper from before the analytic-renderer update.
# Reloading the wrapper also replaces its cached extension loader.
import methods.affinegs.accelerators.gaussian_renderer.renderer as gs_cuda_renderer
importlib.reload(gs_cuda_renderer)
import methods.affinegs.accelerators.gaussian_renderer as gs_cuda_package
importlib.reload(gs_cuda_package)
import methods.affinegs.gaussian_flow.renderer as gs_renderer
importlib.reload(gs_renderer)
import methods.affinegs.gaussian_flow.config as gs_config
importlib.reload(gs_config)
import methods.affinegs.gaussian_flow.model as gs_model
importlib.reload(gs_model)
import methods.affinegs.gaussian_flow.training as gs_training
importlib.reload(gs_training)
import methods.method as method_api
importlib.reload(method_api)
GaussianFlowConfig = gs_config.GaussianFlowConfig
run_gaussian_flow = gs_training.run_gaussian_flow
ilk_vg, inr_vg, of_vg = method_api.ilk_vg, method_api.inr_vg, method_api.of_vg
from methods.inr.inr import MLP_Conv

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for these experiments.')

DEVICE = torch.device('cuda')
BASE_SEED = 42
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {torch.cuda.get_device_name(0)}')

Using device: cuda
Project root: /media/newdisk/junjie/2dGS-representation
Device: NVIDIA GeForce RTX 3090


In [ ]:
def set_all_seeds(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    torch.cuda.manual_seed_all(value)


def count_trainable_parameters(model):
    return int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))

## Experiment 1: Table 3 Efficiency Benchmark

JHTDB uses the first EXP2 sample (3 ms, 16 bins); EBIV follows EXP3 (10 ms, 21 bins). GS uses the shared pruning threshold 0.12 and fused native-resolution CUDA rendering. INR is not chunked; its final output queries velocity only, while training derivative regularization is unchanged. Both methods disable autograd during final inference.

CUDA/JIT warmup and input generation/loading are outside measurement. Input upload is inside. Every method resets the peak counter exactly once, after checking that no live CUDA tensors remain. CPU baselines use CPU preprocessing and still have their CUDA peak checked. Results are released between methods.

Only the allocated peak through the end of training is reported. Final rendering is excluded from the reported memory, but included in runtime. Classical methods have no training-memory entry.


In [ ]:
# Shared GS settings for Table 3 and the count study.
table3_synthetic_cfg = GaussianFlowConfig(
    H=256, W=256, verbose=False, prune_opacity_thresh=0.12,
    render_tile_batch_size=32,
)
table3_real_cfg = GaussianFlowConfig(
    H=720, W=1280, init_n=500, w_div=100, verbose=False, tile_size=80,
    init_scale_h=50.0, init_scale_w=35.0,
    cmax_sigma=0.6, cmax_kernel_size=5, lr_j=None,
    enable_jacobian=True, jacobian_mode='full',
    iterations=300, residual_block_size=40,
    log_scale_min=np.log(8), phase=[50, 200, 250],
    render_tile_batch_size=32,
)


def to_event_tensor(events):
    if isinstance(events, torch.Tensor):
        return events.detach().to(device='cpu', dtype=torch.float32).contiguous()
    return torch.as_tensor(events, dtype=torch.float32, device='cpu').contiguous()


def make_table3_synthetic_events():
    set_all_seeds(BASE_SEED)
    flow = FlowManager(
        mode='data',
        data_path=str(PROJECT_ROOT / 'data/datasets/JHTDB_isotropic1024_hd_00001_flow.npz'),
    )
    cfg = config_events()
    cfg.height, cfg.width = 256, 256
    cfg.time_length = 3.0
    cfg.density = 0.05
    cfg.d = 0.5
    cfg.device = 'cuda:0'
    events, _ = EventManager(
        source_type='simulator', path=None, cfg=cfg, flow_func=flow.evaluate
    ).load()
    events = to_event_tensor(events)
    return events[(events[:, 2] > 0.0) & (events[:, 2] < method_api.EVENT_WINDOW_MS)]


def load_table3_recorded_events(path, source_type='real', window=(10.0, 20.0)):
    start_ms, end_ms = map(float, window)
    events, _ = EventManager(source_type=source_type, path=str(path)).load()
    return to_event_tensor(events[(events[:, 2] > start_ms) & (events[:, 2] < end_ms)])


_table3_warmed_up = False


def warmup_table3():
    global _table3_warmed_up
    if _table3_warmed_up:
        return
    # Initialize the actual CUDA kernels outside the measured task.
    warm_events = torch.rand(96, 4, device=DEVICE)
    warm_events[:, :2] *= 31
    warm_events[:, 2] *= 3
    warm_events[:, 3] = (warm_events[:, 3] > 0.5).float()
    warm_result = run_gaussian_flow(
        warm_events, GaussianFlowConfig(
            H=32, W=32, init_n=3, iterations=1, enable_density_control=False,
            render_tile_batch_size=32, device=str(DEVICE),
        ), flowgt=None,
    )
    warm_model = MLP_Conv(ne=64, beta=100).to(DEVICE)
    warm_xs = torch.rand(1, 2, 8, 8, device=DEVICE)
    warm_values = warm_model.forward_manual_grad(warm_xs)
    sum(value.square().mean() for value in warm_values).backward()
    del warm_events, warm_result, warm_model, warm_xs, warm_values
    torch.cuda.synchronize(DEVICE)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize(DEVICE)
    _table3_warmed_up = True


def measure_call(method_name, events_cpu, *, H, W, num_bins, gaussian_cfg, seed):
    if events_cpu.device.type != 'cpu':
        raise ValueError('Table 3 requires CPU events before measurement.')
    warmup_table3()
    torch.cuda.synchronize(DEVICE)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize(DEVICE)
    baseline = torch.cuda.memory_allocated(DEVICE)
    if baseline:
        raise RuntimeError(
            f'{baseline / 2**20:.2f} MiB of live CUDA tensors remain before measurement; '
            'release them or restart the kernel. The baseline is not subtracted.'
        )
    # Restore the same per-method initialization after warmup.
    set_all_seeds(seed)
    metadata = {'parameters': None, 'final_gaussians': None, 'training_peak_bytes': None}
    original_inr_train = method_api.train_func
    original_gs_train = gs_training.train_gaussian_flow

    def train_inr_with_checkpoint(*args, **kwargs):
        model = original_inr_train(*args, **kwargs)
        metadata['training_peak_bytes'] = torch.cuda.max_memory_allocated(DEVICE)
        metadata['parameters'] = count_trainable_parameters(model)
        return model

    def train_gs_with_checkpoint(*args, **kwargs):
        model, history = original_gs_train(*args, **kwargs)
        metadata['training_peak_bytes'] = torch.cuda.max_memory_allocated(DEVICE)
        metadata['parameters'] = count_trainable_parameters(model)
        metadata['final_gaussians'] = int(model.num_gaussians)
        metadata['render_backend'] = model.render_backend
        metadata['cmax_backend'] = 'cuda' if sys.modules['methods.affinegs.cmax']._HAS_FUSED_WARP_IWE else 'torch'
        return model, history

    # One reset before upload/preprocessing; training checkpoints only READ this counter.
    torch.cuda.reset_peak_memory_stats(DEVICE)
    started = time.perf_counter()
    with ExitStack() as stack:
        if method_name == 'INR':
            stack.enter_context(patch.object(method_api, 'train_func', train_inr_with_checkpoint))
            uploaded = events_cpu.to(DEVICE)
            result = inr_vg(uploaded, H=H, W=W, event_window_ms=None, num_bins=num_bins)
            # INR returns only velocity by default; optional derivative slots are None.
            physical_fields = result[:2]
        elif method_name == 'AffineGS':
            stack.enter_context(patch.object(gs_training, 'train_gaussian_flow', train_gs_with_checkpoint))
            uploaded = events_cpu.to(DEVICE)
            result = run_gaussian_flow(events=uploaded, cfg=gaussian_cfg, flowgt=None)
            physical_fields = (result.flow[0], result.flow[1], result.div, result.vorticity)
        elif method_name in ('ILK-VG', 'OF-VG'):
            estimator = ilk_vg if method_name == 'ILK-VG' else of_vg
            result = estimator(events_cpu, H=H, W=W, event_window_ms=None, num_bins=num_bins)
            physical_fields = result
        else:
            raise ValueError(f'Unknown method: {method_name}')
        for field in physical_fields:
            if isinstance(field, torch.Tensor):
                if field.device.type != 'cpu' or field.requires_grad or field.grad_fn is not None:
                    raise RuntimeError('Final fields must be detached CPU outputs.')
        fields_cpu = np.stack([
            field.numpy() if isinstance(field, torch.Tensor) else field
            for field in physical_fields
        ])
        del result, physical_fields
        if method_name in ('INR', 'AffineGS'):
            del uploaded
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - started
    peak_allocated = torch.cuda.max_memory_allocated(DEVICE)
    peak_reserved = torch.cuda.max_memory_reserved(DEVICE)
    end_allocated = torch.cuda.memory_allocated(DEVICE)
    expected_channels = 2 if method_name == 'INR' else 4
    if fields_cpu.shape != (expected_channels, H, W) or not np.isfinite(fields_cpu).all():
        raise RuntimeError('Incomplete or non-finite final physical fields.')
    if end_allocated:
        raise RuntimeError('CUDA tensors remain attached to the returned result.')
    metadata.update({
        'elapsed_s': elapsed,
        'baseline_bytes': baseline,
        'peak_allocated_bytes': peak_allocated,
        'peak_reserved_bytes': peak_reserved,
        'end_allocated_bytes': end_allocated,
    })
    return metadata, fields_cpu


TABLE3_CASES = [
    {
        'Data': 'JHTDB-isotropic', 'Resolution': '256×256', 'H': 256, 'W': 256,
        'Window': '0–3 ms', 'Num Bins': method_api.TEMPORAL_NUM_BINS,
        'Gaussian Config': table3_synthetic_cfg, 'Seed': BASE_SEED,
        'Loader': lambda: make_table3_synthetic_events(),
    },
    {
        'Data': 'EBIV', 'Resolution': '1280×720', 'H': 720, 'W': 1280,
        'Window': '10–20 ms', 'Num Bins': 21,
        'Gaussian Config': table3_real_cfg, 'Seed': 2026,
        'Loader': lambda: load_table3_recorded_events(
            PROJECT_ROOT / 'data/real_old/ebiv_wallflow4_dense.h5'
        ),
    },
]
TABLE3_METHODS = ('ILK-VG', 'OF-VG', 'INR', 'AffineGS')


In [ ]:
table3_rows = []

for case in TABLE3_CASES:
    events = case['Loader']()
    H, W = case['H'], case['W']
    print(f"{case['Data']}: {len(events)} CPU events")

    for method_name in TABLE3_METHODS:
        measured, method_result = measure_call(
            method_name, events, H=H, W=W, num_bins=case['Num Bins'],
            gaussian_cfg=case['Gaussian Config'], seed=case['Seed'],
        )
        training_peak = measured['training_peak_bytes']
        table3_rows.append({
            'Method': method_name,
            'Data': case['Data'],
            'Resolution': case['Resolution'],
            'Time Window': case['Window'],
            'Temporal Bins': case['Num Bins'] if method_name != 'AffineGS' else None,
            'Events': int(len(events)),
            'Time (s)': measured['elapsed_s'],
            'Learnable Params': measured['parameters'],
            'Final Gaussians': measured['final_gaussians'],
            'GS Renderer': measured.get('render_backend', '—'),
            'CMax Backend': measured.get('cmax_backend', '—'),
            'Peak through Training (MiB)': None if training_peak is None else training_peak / 2**20,
            'Throughput (MPix/s)': (H * W / 1e6) / measured['elapsed_s'],
        })
        del method_result, measured
        gc.collect()
        torch.cuda.empty_cache()

    del events
    gc.collect()
    torch.cuda.empty_cache()

table3_results = pd.DataFrame(table3_rows)
display(
    table3_results.style
    .hide_index()
    .format({
        'Temporal Bins': lambda value: '—' if pd.isna(value) else f'{int(value)}',
        'Events': '{:,}',
        'Time (s)': '{:.3f}',
        'Learnable Params': lambda value: '—' if pd.isna(value) else f'{int(value):,}',
        'Final Gaussians': lambda value: '—' if pd.isna(value) else f'{int(value)}',
        'Peak through Training (MiB)': lambda value: '—' if pd.isna(value) else f'{value:.1f}',
        'Throughput (MPix/s)': '{:.3f}',
    })
    .set_caption('Table 3 — allocated memory through training; runtime includes final rendering')
)


Method,Data,Resolution,Time Window,Temporal Bins,Events,Time (s),Learnable Params,Final Gaussians,GS Renderer,CMax Backend,Peak through Training (MiB),Throughput (MPix/s)
ILK-VG,JHTDB-isotropic,256×256,0–3 ms,16.000000,224683,1.110,—,—,—,—,—,0.059045
OF-VG,JHTDB-isotropic,256×256,0–3 ms,16.000000,224683,0.176,—,—,—,—,—,0.372890
INR,JHTDB-isotropic,256×256,0–3 ms,16.000000,224683,4.172,"29,122",—,—,—,217.9,0.015707
AffineGS,JHTDB-isotropic,256×256,0–3 ms,nan,224683,0.657,"3,600",300,cuda,cuda,316.1,0.099731
ILK-VG,EBIV,1280×720,10–20 ms,21.000000,410697,2.990,—,—,—,—,—,0.308277
OF-VG,EBIV,1280×720,10–20 ms,21.000000,410697,1.465,—,—,—,—,—,0.628959
INR,EBIV,1280×720,10–20 ms,21.000000,410697,42.802,"29,122",—,—,—,3014.9,0.021532
AffineGS,EBIV,1280×720,10–20 ms,nan,410697,1.409,"5,160",430,cuda,cuda,590.0,0.653986


In [ ]:
# Report only allocated memory through training; exclude final rendering from memory.
table3_paper = table3_results.copy()
table3_paper['_method_order'] = table3_paper['Method'].map(
    {name: index for index, name in enumerate(TABLE3_METHODS)}
)
table3_paper['_resolution_order'] = table3_paper['Resolution'].map(
    {case['Resolution']: index for index, case in enumerate(TABLE3_CASES)}
)
table3_paper = table3_paper.sort_values(['_method_order', '_resolution_order'])
table3_paper['Method'] = table3_paper['Method'].replace({
    'INR': 'INR-based', 'AffineGS': 'AffineGS (Ours)',
})
table3_paper['Peak through Training (MB)'] = table3_paper['Peak through Training (MiB)'] * (2**20 / 1e6)
table3_paper = table3_paper.rename(columns={'Resolution': 'Resolution (px)'})[
    ['Method', 'Resolution (px)', 'Time (s)', 'Learnable Params',
     'Peak through Training (MB)']
].set_index(['Method', 'Resolution (px)'])

display(
    table3_paper.style
    .format({
        'Time (s)': '{:.1f}',
        'Learnable Params': lambda value: '—' if pd.isna(value) else f'{value / 1000:.1f}K',
        'Peak through Training (MB)': lambda value: '—' if pd.isna(value) else f'{value:.1f}',
    })
    .set_caption('Table 3. Computational efficiency and model complexity')
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center')]},
        {'selector': 'th.row_heading', 'props': [('vertical-align', 'middle')]},
        {'selector': 'td', 'props': [('text-align', 'right')]},
    ])
)
print('Reported memory: CUDA allocated peak from CPU input through training, in decimal MB; final rendering is excluded. Runtime includes the complete call and CPU outputs.')
print('GS: fused native-resolution CUDA rendering. INR: full-image velocity inference with no autograd; final analytic derivative maps disabled, network and training unchanged. CPU-method memory is CUDA-only, not CPU RAM.')


<Saved EXP_OTHER table; restored without retraining>

## Experiment 2: Gaussian Count after Density Control

Synthetic flows use five distinct files, 300 initial Gaussians, 3 ms windows, and the local pruning threshold of 0.12. EBIV and Water retain the EXP3 configuration and use five non-overlapping 10 ms windows with 500 initial Gaussians. Final counts are the rounded means of five samples.

In [ ]:
COUNT_DATA_ROOT = Path('/media/newdisk/datasets_piv/piv_cai/train')
COUNT_SYNTHETIC_SPECS = [
    ('JHTDB', 'JHTDB_isotropic1024_hd'),
    ('DNS', 'DNS_turbulence'),
    ('SQG', 'SQG'),
]
COUNT_REAL_SPECS = [
    ('EBIV', 'real', PROJECT_ROOT / 'data/real_old/ebiv_wallflow4_dense.h5'),
    ('Water', 'evk5', PROJECT_ROOT / 'data/water.hdf5'),
]
COUNT_REAL_WINDOWS_MS = [
    (10.0, 20.0), (20.0, 30.0), (30.0, 40.0),
    (40.0, 50.0), (50.0, 60.0),
]


def make_count_synthetic_events(flow):
    cfg = config_events()
    cfg.height, cfg.width = flow.field.h, flow.field.w
    cfg.time_length = 3.0
    cfg.d = 0.5
    cfg.density = 0.05
    cfg.device = 'cuda:0'
    events, (vel_h, vel_w) = EventManager(
        source_type='simulator', path=None, cfg=cfg, flow_func=flow.evaluate
    ).load()
    events = events[(events[:, 2] > 0.0) & (events[:, 2] < 3.0)]
    height, width = int(vel_h.shape[-2]), int(vel_h.shape[-1])
    del vel_h, vel_w
    return events, height, width


def train_and_count_gaussians(events, cfg, seed):
    set_all_seeds(seed)
    result = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
    final_count = int(result.model.num_gaussians)
    del result
    gc.collect()
    torch.cuda.empty_cache()
    return final_count


count_synthetic_cfg = replace(table3_synthetic_cfg)
count_real_cfg = replace(table3_real_cfg)

In [ ]:
count_synthetic_rows = []

for flow_index, (flow_name, prefix) in enumerate(COUNT_SYNTHETIC_SPECS):
    for file_index in range(1, 6):
        file_path = COUNT_DATA_ROOT / f'{prefix}_{file_index:05d}.npz'
        if not file_path.is_file():
            raise FileNotFoundError(file_path)

        event_seed = BASE_SEED + 10000 * flow_index + file_index
        train_seed = event_seed + 100000
        set_all_seeds(event_seed)
        flow = FlowManager(mode='data', data_path=str(file_path))
        events, height, width = make_count_synthetic_events(flow)
        del flow

        cfg = replace(count_synthetic_cfg, H=height, W=width)
        final_count = train_and_count_gaussians(events, cfg, train_seed)
        count_synthetic_rows.append({
            'Flow': flow_name,
            'Sample': f'{file_index:05d}',
            'Initial Gaussians': cfg.init_n,
            'Final Gaussians': final_count,
        })
        del events

count_synthetic_detail = pd.DataFrame(count_synthetic_rows)
display(
    count_synthetic_detail.style
    .hide_index()
    .set_caption('Synthetic samples')
)

Flow,Sample,Initial Gaussians,Final Gaussians
JHTDB,00001,300,296
JHTDB,00002,300,293
JHTDB,00003,300,292
JHTDB,00004,300,291
JHTDB,00005,300,302
DNS,00001,300,321
DNS,00002,300,302
DNS,00003,300,307
DNS,00004,300,320
DNS,00005,300,337


In [ ]:
count_real_rows = []

for flow_name, source_type, file_path in COUNT_REAL_SPECS:
    if not file_path.is_file():
        raise FileNotFoundError(file_path)
    raw_events, _ = EventManager(source_type=source_type, path=str(file_path)).load()

    for start_ms, end_ms in COUNT_REAL_WINDOWS_MS:
        mask = (raw_events[:, 2] > start_ms) & (raw_events[:, 2] < end_ms)
        events = torch.as_tensor(raw_events[mask], dtype=torch.float32, device='cpu')
        if len(events) == 0:
            raise RuntimeError(f'Empty event window: {flow_name} {start_ms:g}–{end_ms:g} ms')

        final_count = train_and_count_gaussians(events, count_real_cfg, seed=2026)
        count_real_rows.append({
            'Flow': flow_name,
            'Sample': f'{start_ms:g}–{end_ms:g} ms',
            'Initial Gaussians': count_real_cfg.init_n,
            'Final Gaussians': final_count,
        })
        del events

    del raw_events
    gc.collect()

count_real_detail = pd.DataFrame(count_real_rows)
display(
    count_real_detail.style
    .hide_index()
    .set_caption('Real-event windows')
)

Flow,Sample,Initial Gaussians,Final Gaussians
EBIV,10–20 ms,500,442
EBIV,20–30 ms,500,446
EBIV,30–40 ms,500,540
EBIV,40–50 ms,500,455
EBIV,50–60 ms,500,442
Water,10–20 ms,500,455
Water,20–30 ms,500,474
Water,30–40 ms,500,488
Water,40–50 ms,500,487
Water,50–60 ms,500,481


In [ ]:
count_all_detail = pd.concat([count_synthetic_detail, count_real_detail], ignore_index=True)
count_flow_order = ['JHTDB', 'DNS', 'SQG', 'EBIV', 'Water']

count_summary_rows = []
for flow_name in count_flow_order:
    rows = count_all_detail[count_all_detail['Flow'] == flow_name]
    mean_final = float(rows['Final Gaussians'].mean())
    count_summary_rows.append({
        'Flow': flow_name,
        'Initial Gaussians': int(rows['Initial Gaussians'].iloc[0]),
        'Final Gaussians': int(math.floor(mean_final + 0.5)),
    })

gaussian_count_table = pd.DataFrame(count_summary_rows)
display(
    gaussian_count_table.style
    .hide_index()
    .set_caption('Gaussian Count after Density Control (mean of five samples)')
)

Flow,Initial Gaussians,Final Gaussians
JHTDB,300,295
DNS,300,317
SQG,300,303
EBIV,500,465
Water,500,477
